# Practical 3: Industrial Electricity Production Analysis
## Dataset: Electric_Production.xls (Monthly Production Index)

---
### 📘 Beginner's Concept: Industrial Demand Forecasting
Electricity production displays cyclical peak demand during winter heating months and summer air conditioning months. We apply decomposition, Holt-Winters Multiplicative modeling, and stationarity differencing.
---

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
from statsmodels.tsa.seasonal import seasonal_decompose

In [ ]:
df = pd.read_csv("Electric_Production.xls")
df.head(5)

In [ ]:
df.shape

In [ ]:
# convert into date time
df["DATE"] = pd.to_datetime(df["DATE"])
df = df.set_index("DATE")
df.head(5)

In [ ]:
sns.lineplot(df)
plt.ylabel("Electricity_Production")
plt.title("Industrial Electricity Production Index")
plt.show()

In [ ]:
# decomposition of the time series - multiplicative model
result = seasonal_decompose(df[["IPG2211A2N"]], model = "multiplicative", period = 12)
result.plot()
plt.show()

In [ ]:
# Perform the Mann-Kendall test
import pymannkendall as mk
mk.original_test(df["IPG2211A2N"])

In [ ]:
# Train test splitting (70% Train, 30% Test)
train_df = df[:int(df.shape[0]*0.7)]
test_df = df[int(df.shape[0]*0.7):]
train_df.head(5)

In [ ]:
# Triple Exponential Smoothing (Holt-Winters Multiplicative)
from statsmodels.tsa.api import ExponentialSmoothing
model_triple_mul = ExponentialSmoothing(train_df["IPG2211A2N"], seasonal_periods = 12, trend = "add", seasonal = "mul")
model_triple_fit_mul = model_triple_mul.fit()
forecast_triple_mul = model_triple_fit_mul.forecast(len(test_df))
print(forecast_triple_mul)

In [ ]:
from sklearn.metrics import mean_absolute_percentage_error
mape = mean_absolute_percentage_error(test_df["IPG2211A2N"], forecast_triple_mul)
print("MAPE Test for Electricity Production:", mape)

In [ ]:
# ADF and KPSS Stationarity Tests
from statsmodels.tsa.stattools import adfuller, kpss
print("ADF p-value:", adfuller(df["IPG2211A2N"])[1])
print("KPSS p-value:", kpss(df["IPG2211A2N"])[1])

In [ ]:
# Combined Seasonal and Non-Seasonal Differencing
sddiff = df["IPG2211A2N"].diff(12).diff().dropna()
print("Differenced ADF p-value:", adfuller(sddiff)[1])
print("Differenced KPSS p-value:", kpss(sddiff)[1])